# Train GPT-2-small (from scratch) on a free cloud GPU

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sikander-Iqbal/LLM_From_Scratch/blob/master/notebooks/train_gpt2_small.ipynb)

Runs the [LLM_From_Scratch](https://github.com/Sikander-Iqbal/LLM_From_Scratch) training pipeline on whatever free GPU this notebook is running on (Kaggle's P100/T4, or Colab's T4).

**Free-tier sessions are time-limited** (Kaggle ~9-12h, Colab ~12h). This notebook is built around that: it trains in one bounded chunk, then tells you exactly what to do to pick up where you left off in the next session, using `train.py`'s `--init_from resume` support.

### How to use this on each platform
- **Kaggle**: File > New Notebook > Import Notebook, paste this file's GitHub URL. Turn on GPU under Settings > Accelerator. Turn on internet access under Settings.
- **Colab**: click the badge above, or File > Open notebook > GitHub, paste the repo URL. Runtime > Change runtime type > GPU.

### Resuming across sessions
Each session starts with a fresh disk. Before running the training cell, upload your previous `ckpt.pt` (see the "Resume from a previous checkpoint" section below) — otherwise it starts a brand new run from scratch.

## 1. Check the GPU we got

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

## 2. Clone the repo and install the (few) extra dependencies

Torch is already preinstalled with the right CUDA build on both Kaggle and Colab — we deliberately don't touch it, just add the small extra libraries the project needs.

In [ ]:
import os

REPO_URL = 'https://github.com/Sikander-Iqbal/LLM_From_Scratch.git'
REPO_DIR = 'LLM_From_Scratch'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL}
%cd {REPO_DIR}
!pip install -q tokenizers datasets tqdm

import torch
print('torch', torch.__version__, '| cuda available:', torch.cuda.is_available())

## 3. (Optional) Resume from a previous checkpoint

If this is your **first** session, skip this and go straight to section 4 — training will start from scratch.

If you're **continuing** a run: upload the `ckpt.pt` you downloaded at the end of your last session.
- **Colab**: run the cell below, it will prompt a file picker.
- **Kaggle**: instead, add your checkpoint as a Kaggle Dataset (Add Data > Upload), then the second cell below copies it in from `/kaggle/input/`.

In [ ]:
# Colab only: uncomment and run to upload ckpt.pt via a file picker
# from google.colab import files
# import shutil
# os.makedirs('checkpoints/gpt2_small', exist_ok=True)
# uploaded = files.upload()
# for name in uploaded:
#     shutil.move(name, 'checkpoints/gpt2_small/ckpt.pt')
# print('checkpoint restored')

In [ ]:
# Kaggle only: uncomment and set DATASET_NAME to whatever you named your uploaded checkpoint dataset
# import shutil
# DATASET_NAME = 'my-gpt2-checkpoint'
# os.makedirs('checkpoints/gpt2_small', exist_ok=True)
# shutil.copy(f'/kaggle/input/{DATASET_NAME}/ckpt.pt', 'checkpoints/gpt2_small/ckpt.pt')
# print('checkpoint restored')

## 4. Prepare the tokenizer + data (skips automatically if already done)

In [ ]:
if not os.path.exists('tokenizer/bpe_tokenizer.json'):
    %run tokenizer/dump_corpus_sample.py --dataset NeelNanda/pile-10k --output tokenizer/corpus_sample.txt
    %run tokenizer/train_tokenizer.py --input tokenizer/corpus_sample.txt --vocab_size 50257
else:
    print('tokenizer already trained, skipping')

if not os.path.exists('data/openwebtext/train.bin'):
    %run data/openwebtext/prepare.py --dataset NeelNanda/pile-10k --num_proc 2
else:
    print('data already tokenized, skipping')

## 5. Train

`--init_from resume` is safe to leave on always: it resumes if `checkpoints/gpt2_small/ckpt.pt` exists (e.g. you restored one in step 3), and falls back to fresh init otherwise... actually it doesn't fall back automatically, so we detect that here.

**Bump `TARGET_MAX_ITERS` up each session** (don't reset it) — the run always trains up to this absolute iteration count, continuing from wherever the restored checkpoint left off. Free GPUs here have no laptop-style thermal/power cap and 16GB VRAM (vs. 8GB), so expect noticeably faster iterations than the laptop run — watch the first few `iter N:` log lines to see your actual ms/iter and budget your session accordingly (see the README's ETA formula).
**If the training cell crashed or errored on a previous attempt, restart the kernel/session before re-running it (Kaggle: Session menu > Restart Session; Colab: Runtime > Restart session) rather than just re-running the cell.** Jupyter keeps a crashed cell's local variables (including the model and its GPU tensors) alive via the exception traceback, so GPU memory from a failed attempt can stay allocated even after the cell finishes. Re-running `%run` in the same still-warm kernel stacks a fresh model on top of that leftover memory and can OOM even at a batch size that worked before. A kernel restart fully clears GPU memory; a plain cell re-run does not.

In [ ]:
TARGET_MAX_ITERS = 500  # raise this each session; never lower it below your last run's iter_num

init_from = 'resume' if os.path.exists('checkpoints/gpt2_small/ckpt.pt') else 'scratch'
print('init_from =', init_from)

# build the full command as a plain Python string (not relying on magic-line {} interpolation)
train_args = (
    f"train.py --data_dir data/openwebtext --out_dir checkpoints/gpt2_small "
    f"--init_from {init_from} "
    f"--n_layer 12 --n_head 12 --n_embd 768 --block_size 1024 "
    f"--batch_size 16 --gradient_accumulation_steps 20 "
    f"--max_iters {TARGET_MAX_ITERS} --eval_interval 100 --dtype bfloat16"
)
get_ipython().run_line_magic('run', train_args)


## 6. Sanity-check the checkpoint by generating some text

In [ ]:
%run sample.py --out_dir checkpoints/gpt2_small --tokenizer_path tokenizer/bpe_tokenizer.json \
    --prompt "The meaning of life is" --max_new_tokens 150

## 7. Save the checkpoint before your session ends

This is the important step — without it, everything trained this session is lost when the runtime recycles.

In [ ]:
# Colab: downloads ckpt.pt to your computer directly
# from google.colab import files
# files.download('checkpoints/gpt2_small/ckpt.pt')

# Kaggle: just click "Save Version" (top right) with "Save & Run All" —
# anything under /kaggle/working (this repo's checkpoints/ folder) is kept as the version's output,
# downloadable from the notebook's Output tab, and reusable as a Dataset input next session.
print('checkpoint at:', os.path.abspath('checkpoints/gpt2_small/ckpt.pt'))